# Task 5 — Content-Based TED Talks Recommendation Engine

**Objective:** create a reusable `recommend_talks(title, top_n=5)` function. TF-IDF is fitted once when the notebook starts; a user choosing another title only requires a similarity lookup.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the project folder from the current Jupyter working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "ted_talks_merged.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ted_talks_merged.csv. "
        "Open Jupyter from the TED_Talks_Recommendation_Ajay project folder."
    )


PROJECT_DIR = find_project_root()
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "ted_talks_merged.csv"
PROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"
print("Project folder:", PROJECT_DIR)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError("Run Task 3 first.")
df = pd.read_csv(PROCESSED_DATA_PATH)
df["combined_text"] = df["combined_text"].fillna("").astype(str)
usable_mask = df["combined_text"].str.strip().ne("")
work_df = df.loc[usable_mask].reset_index(drop=True)

vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2), min_df=2)
tfidf_matrix = vectorizer.fit_transform(work_df["combined_text"])

# A title lookup makes the public function easier to use than a numeric row index.
title_to_index = {str(title).strip().casefold(): index for index, title in enumerate(work_df["title"])}
print("Model fitted once.")
print("Talks indexed:", len(title_to_index))
print("TF-IDF shape:", tfidf_matrix.shape)

Project folder: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay
Model fitted once.
Talks indexed: 2550
TF-IDF shape: (2550, 20000)


## 1. Reusable recommendation function

In [3]:
def recommend_talks(title, top_n=5):
    """Return the most textually similar TED Talks for an exact title match."""
    if not isinstance(title, str) or not title.strip():
        raise ValueError("Please enter a non-empty TED Talk title.")
    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    key = title.strip().casefold()
    if key not in title_to_index:
        raise ValueError(f"TED Talk title not found: {title!r}. Check the spelling or copy a title from the dataset.")

    selected_index = title_to_index[key]
    scores = cosine_similarity(tfidf_matrix[selected_index], tfidf_matrix).ravel()
    scores[selected_index] = -1  # a talk must not recommend itself
    count = min(top_n, len(work_df) - 1)
    best_indices = np.argsort(scores)[::-1][:count]

    columns = [column for column in ["talk_id", "title", "main_speaker", "url"] if column in work_df.columns]
    result = work_df.iloc[best_indices][columns].copy()
    result["similarity_score"] = scores[best_indices]
    return result.reset_index(drop=True)

## 2. Test the recommender with different titles

In [4]:
for selected_title in work_df["title"].head(3).tolist():
    print("\nSelected talk:", selected_title)
    display(recommend_talks(selected_title, top_n=5))


Selected talk: Do schools kill creativity?


,talk_id,title,main_speaker,url,similarity_score
0,Ken Robinson: How to escape education's death ...,How to escape education's death valley,Ken Robinson,https://www.ted.com/talks/ken_robinson_how_to_...,0.364297
1,Ken Robinson: Bring on the learning revolution!,Bring on the learning revolution!,Ken Robinson,https://www.ted.com/talks/sir_ken_robinson_bri...,0.356238
2,Ze Frank: Nerdcore comedy,Nerdcore comedy,Ze Frank,https://www.ted.com/talks/ze_frank_s_nerdcore_...,0.268711
3,Ricardo Semler: How to run a company with (alm...,How to run a company with (almost) no rules,Ricardo Semler,https://www.ted.com/talks/ricardo_semler_how_t...,0.265060
4,Sakena Yacoobi: How I stopped the Taliban from...,How I stopped the Taliban from shutting down m...,Sakena Yacoobi,https://www.ted.com/talks/sakena_yacoobi_how_i...,0.264126



Selected talk: Averting the climate crisis


,talk_id,title,main_speaker,url,similarity_score
0,Al Gore: New thinking on the climate crisis,New thinking on the climate crisis,Al Gore,https://www.ted.com/talks/al_gore_s_new_thinki...,0.308842
1,Ted Halstead: A climate solution where all sid...,A climate solution where all sides can win,Ted Halstead,https://www.ted.com/talks/ted_halstead_a_clima...,0.273597
2,Al Gore: The case for optimism on climate change,The case for optimism on climate change,Al Gore,https://www.ted.com/talks/al_gore_the_case_for...,0.260947
3,Alice Bows-Larkin: Climate change is happening...,Climate change is happening. Here's how we adapt,Alice Bows-Larkin,https://www.ted.com/talks/alice_bows_larkin_we...,0.237302
4,Tom Rielly: A comic sendup of TED2006,A comic sendup of TED2006,Tom Rielly,https://www.ted.com/talks/tom_rielly_delivers_...,0.226043



Selected talk: Simplicity sells


,talk_id,title,main_speaker,url,similarity_score
0,David Pogue: Cool tricks your phone can do,Cool tricks your phone can do,David Pogue,https://www.ted.com/talks/david_pogue_on_cool_...,0.266674
1,Ze Frank: Nerdcore comedy,Nerdcore comedy,Ze Frank,https://www.ted.com/talks/ze_frank_s_nerdcore_...,0.264832
2,David Carson: Design and discovery,Design and discovery,David Carson,https://www.ted.com/talks/david_carson_on_design,0.257508
3,John Maeda: Designing for simplicity,Designing for simplicity,John Maeda,https://www.ted.com/talks/john_maeda_on_the_si...,0.239136
4,Dan Bricklin: Meet the inventor of the electro...,Meet the inventor of the electronic spreadsheet,Dan Bricklin,https://www.ted.com/talks/dan_bricklin_meet_th...,0.235178


## 3. Invalid title handling

In [5]:
try:
    recommend_talks("This TED Talk Title Does Not Exist", top_n=5)
except ValueError as error:
    print("Handled invalid input:", error)

Handled invalid input: TED Talk title not found: 'This TED Talk Title Does Not Exist'. Check the spelling or copy a title from the dataset.


## 4. Why the model is reusable

The vectorizer and TF-IDF matrix are created once in the setup cell. `recommend_talks()` does not fit the vectorizer again; it only finds the selected talk's row, calculates similarity against the existing matrix, removes the selected talk, and returns the highest-scoring results.

**Limitation:** this is content-based similarity, not a personalized user-preference model. A user can receive different recommendations after choosing a different talk, but the system does not yet learn from ratings or watch history.